In [90]:
# Libraries
import numpy as np
import pandas as pd
import deap
import random

from collections import defaultdict
from deap import base, creator, tools, algorithms

In [91]:
# Import prediction data
df = pd.read_csv("Results/RUL_predictions.csv")
RUL = dict(zip(df.engine_id, df.RUL_predicted))

# Set random seed

random.seed(33)

In [92]:
# Various helper functions and other structures
def get_engine_due_date(engine_id, RUL):
    '''Safety due date is ts = t1 + Rj - 1, and t1 = 1, so ts = Rj'''
    return int(RUL[engine_id])

def get_maintenance_duration(engine_id, team_type):
    '''Returns the days to perform maintenance for the input team type and engine ID'''
    # First get the days for team A
    mu_A = None
    if 1 <= engine_id <= 20:
        mu_A = 5
    elif 21 <= engine_id <= 55:
        mu_A = 3
    elif 56 <= engine_id <= 80:
        mu_A = 4
    else:
        mu_A = 5

    # If team type A, return mu_A. Otherwise, use mu_A to calculate team B's days.
    if team_type == "A":
        return mu_A
    elif team_type == "B":
        if 1 <= engine_id <= 25:
            return mu_A - 1
        elif 26 <= engine_id <= 70:
            return mu_A + 3
        else:
            return mu_A + 2
    else:
        raise ValueError("get_maintenance_duration() received invalid team type!")
     
def get_penalty_cost(engine_id, completion_date, due_date):
    '''Computes the penalty cost given the completion and due dates of an engine'''

    # In case completed in time, no penalty
    if completion_date <= due_date:
        return 0

    # Get the cj value for the engine's id range
    cj = None
    if 1 <= engine_id <= 25:
        cj = 4
    elif 26 <= engine_id <= 45:
        cj = 2
    elif 46 <= engine_id <= 75:
        cj = 5
    else:
        cj = 6

    # Compute total penalty value
    # Penalty is charged for every late day after due date
    penalty = 0

    for day in range(due_date + 1, completion_date + 1):
        delay = day - due_date
        daily_penalty = cj * (delay ** 2)
        penalty += min(daily_penalty, 250)  # Add capped daily penalty

    return penalty  # Return total accumulated penalty cost

# Team type mapping
TEAM_TYPES = {
    1: "A",
    2: "B",
    3: "A",
    4: "B"
}

In [93]:
### MAIN GA FUNCTIONS ###

# Encoding the GA individual (chromosomes)
# General idea: [(engine, team, start_day), (engine, team, start_day), ...]
# So one gene is one assignment of maintenance; this structure should be flexible enough for all constraints and operations

# Build schedules for teams
def build_schedule(individual):
    team_jobs = defaultdict(list)

    for engine, team, start in individual:
        team_jobs[team].append((engine, start))

    return team_jobs

def is_feasible(team_jobs):
    '''Returns whether or not a provided (individual's) schedule is feasible given the constraints'''
    for team, jobs in team_jobs.items():
        jobs = sorted(jobs, key=lambda x: x[1])
        current_time = 0

        for engine, start in jobs:
            duration = get_maintenance_duration(engine, TEAM_TYPES[team])
            if start < current_time: # If the start time of a job is before the current time, the schedule is obviously not feasible.
                return False
            if start + duration - 1 > 30: # Planning horizon of 30, so job duration cannot be longer than this.
                return False
            current_time = start + duration

    return True # If no issues caught during the loops, the schedule is valid.

def evaluate(individual, RUL):
    '''
    Calculates the total penalty for a given individual.
    Trailing commas in outputs are there for DEAP format.
    '''
    team_jobs = build_schedule(individual)

    if not is_feasible(team_jobs):
        return (1e9,) # Arbitrarily large penalty for invalid individuals
    
    maintained_engines = set()
    total_penalty = 0

    # Calculate penalties for maintained engines
    for team, jobs in team_jobs.items():
        for engine, start in jobs:
            duration = get_maintenance_duration(engine, TEAM_TYPES[team])
            completion = start + duration - 1
            due_date = get_engine_due_date(engine, RUL)
            total_penalty += get_penalty_cost(engine, completion, due_date)
            maintained_engines.add(engine)

    # Calculate penalties for non-maintained engines
    for engine in range(1, 101):
        if engine not in maintained_engines:
            due = get_engine_due_date(engine, RUL)
            total_penalty += get_penalty_cost(engine, 30, due)

    return (total_penalty,)

def custom_mutation(individual, prob=0.3):
    '''
    Combines multiple small mutations into one to ensure all parts of the chromosome can be mutated.
    This includes start time, team type, engine id, and chromosome length (amount of engines maintained).
    '''
    # We don't re set the seed because putting the seed inside mutation makes the same mutation pattern repeat every time, which hurts the GA


    # Start time mutation
    for i in range(len(individual)):
        if random.random() < prob:
            engine, team, start = individual[i]
            # Shift the start time by a random amount within reasonable range
            shift = random.randint(-3, 3)
            new_start = max(1, min(30, start + shift))
            individual[i] = (engine, team, new_start)

    # Team type mutation
    for i in range(len(individual)):
        if random.random() < prob:
            engine, team, start = individual[i]
            # Select a new team from the available 4 teams
            new_team = random.randint(1, 4)
            individual[i] = (engine, new_team, start)

    # Engine id mutation
    for i in range(len(individual)):
        if random.random() < prob:
            engine, team, start = individual[i]
            # Select a new team from the 100 available engines
            # Has a small (1/100) chance to select the same engine; should be fine given the small chance though.
            new_engine = random.randint(1, 100)
            individual[i] = (new_engine, team, start)

    # Chromosome length (# of engines maintained) mutation
    # Chance to remove engine
    if random.random() < 0.3 and len(individual) > 1: # Ensure we do not end up with chromosomes of 0 length (no engines maintained)
        individual.pop(random.randrange(len(individual)))

    # Chance to add engine
    elif random.random() > 0.7:
        individual.append((
            random.randint(1, 100),
            random.randint(1, 4),
            random.randint(1, 30)
        ))

    # Return mutated individual
    return (individual,)

def custom_crossover(parent_1, parent_2):
    '''
    Custom crossover function with the goal of keeping schedules reasonably intact.
    To that end, children inherit subsets (~half) of parent schedules, rather than single point crossover or similar.
    '''
    child_1, child_2 = [], []

    split_1 = set(random.sample(range(len(parent_1)), k=len(parent_1)//2))
    split_2 = set(random.sample(range(len(parent_2)), k=len(parent_2)//2))

    # Child 1 creation
    child_1.extend(parent_1[i] for i in split_1)
    # Make sure we do not duplicate engine maintenance
    maintained_engines_1 = {g[0] for g in child_1}
    for gene in parent_2:
        if gene[0] not in maintained_engines_1:
            child_1.append(gene)

    # Child 2 creation
    child_2.extend(parent_2[i] for i in split_2)
    # Make sure we do not duplicate engine maintenance
    maintained_engines_2 = {g[0] for g in child_2}
    for gene in parent_1:
        if gene[0] not in maintained_engines_2:
            child_2.append(gene)

    # Overwrite parents with children
    parent_1[:] = child_1
    parent_2[:] = child_2

    return parent_1, parent_2

In [94]:
# DEAP Setup
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)
model = base.Toolbox()

# Generate individuals
def create_individual():
    n_jobs = random.randint(10, 40) # Ensure varied but reasonable schedule lengths
    indiv = []
    for i in range(n_jobs):
        engine = random.randint(1, 100)
        team = random.randint(1, 4)
        start = random.randint(1, 30)
        indiv.append((engine, team, start))

    return creator.Individual(indiv)

model.register("individual", create_individual)
model.register("population", tools.initRepeat, list, model.individual)

# Placeholder operators (TODO: better functions and run the model)
model.register("evaluate", evaluate, RUL=RUL)
model.register("mate", custom_crossover)
model.register("mutate", custom_mutation, prob=0.3)
model.register("select", tools.selTournament, tournsize=3)

In [95]:
# GA Running code
def run_ga(RUL, population_size=80, generations=60, cxpb=0.8, mutpb=0.4): #TODO: justify and/or change default numbers
    '''
    Runs the genetic algorithm using the functions defined previously.
    '''
    # Intialize population
    population = model.population(n=population_size)
    # Keep track of best solution
    track_best_sol = tools.HallOfFame(1)
    # Track useful statistics
    stats = tools.Statistics(lambda indiv: indiv.fitness.values[0])
    stats.register("avg", lambda x: sum(x)/len(x))
    stats.register("min", min)
    stats.register("max", max)

    # Setup algorithm
    population, log = algorithms.eaSimple(population, model, cxpb=cxpb, mutpb=mutpb, ngen=generations, stats=stats, halloffame=track_best_sol, verbose=True)

    # Save final best solution
    best = track_best_sol[0]

    return best, log

# Run GA
# best_solution, log = run_ga(RUL)
# print(best_solution, log)

In [96]:
def repair(individual, T=30):
    """
    Repairs an individual in-place so all constraints are satisfied:
      1. No engine is scheduled more than once (drop duplicates)
      2. Each job fits within the planning horizon [1, T]
      3. No two jobs of the same team overlap in time

    Parameters
    ----------
    individual : DEAP individual — list of (engine_id, team_id, start_day) tuples
    T          : planning horizon in days (default 30)
    """

    # Remove duplicate engine assignments (keep first occurrence)
    seen = set()
    unique_genes = []
    for gene in individual:
        engine_id = gene[0]
        if engine_id not in seen:
            seen.add(engine_id)
            unique_genes.append(gene)
    individual[:] = unique_genes

    # Fix each gene so it fits within the horizon
    valid_genes = []
    for engine_id, team_id, start_day in individual:
        team_id   = max(1, min(4, int(team_id)))
        start_day = max(1, int(start_day))

        duration = get_maintenance_duration(engine_id, TEAM_TYPES[team_id])

        # Push start back if job would exceed the horizon
        if start_day + duration - 1 > T:
            start_day = T - duration + 1

        # If it still doesn't fit (duration > T), drop this engine entirely
        if start_day < 1:
            continue

        valid_genes.append((engine_id, team_id, start_day))

    individual[:] = valid_genes

    # Resolve time overlaps within each team
    for team_id in (1, 2, 3, 4):
        # Skip None entries left by previous teams in this same pass
        team_idx = [i for i, g in enumerate(individual) if g is not None and g[1] == team_id]
        team_idx.sort(key=lambda i: individual[i][2])

        next_free = 1

        for i in team_idx:
            engine_id, t, start_day = individual[i]
            duration = get_maintenance_duration(engine_id, TEAM_TYPES[t])

            start_day = max(start_day, next_free)
            end_day   = start_day + duration - 1

            if end_day > T:
                individual[i] = None
            else:
                individual[i] = (engine_id, t, start_day)
                next_free = end_day + 1

    individual[:] = [g for g in individual if g is not None]

    return individual

In [97]:
# DEAP toolbox setup
# Toolbox that wires up operators and repair

# Guard against re-creation errors when re-running this cell
if "FitnessMin" not in dir(creator):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if "Individual" not in dir(creator):
    creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("individual", create_individual)   # Stefan's Cell 4 function
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate",   evaluate, RUL=RUL)   # Stefan's Cell 3 function
toolbox.register("select",     tools.selTournament, tournsize=3)
toolbox.register("mate",       custom_crossover)    # Stefan's Cell 3 function
toolbox.register("mutate",     custom_mutation, prob=0.3)  # Stefan's Cell 3 function

In [98]:
# Time-capped GA loop (5-minute hard stop)

import time

T          = 30
N_ENGINES  = 100
POP_SIZE   = 100
CX_PROB    = 0.70
MUT_PROB   = 0.20
MAX_SECONDS = 300  # 5-minute cap

# Candidate engines: those that can fail within the planning horizon
candidates = [eid for eid in range(1, N_ENGINES + 1) if RUL.get(eid, 999) < T]
print(f"Candidate engines (RUL < {T}): {len(candidates)} → {candidates}")


def run_ga(rul_input=RUL, seed=None):
    """
    Runs the GA for up to MAX_SECONDS seconds.

    Parameters
    ----------
    rul_input : dict {engine_id: RUL} — pass consultancy dict for Task 2.3
    seed      : optional int for reproducibility

    Returns
    -------
    best_ind  : best individual found
    best_cost : its total penalty (float)
    history   : list of best-fitness per generation
    """
    if seed is not None:
        random.seed(seed)

    # Re-register evaluate with the correct RUL dict for this run
    toolbox.register("evaluate", evaluate, RUL=rul_input)

    # Initialise and evaluate the starting population
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        repair(ind)                              # ensure feasibility from the start
        ind.fitness.values = toolbox.evaluate(ind)

    history    = []
    start_time = time.time()
    gen        = 0

    while time.time() - start_time < MAX_SECONDS:
        gen += 1

        # 1. Selection
        offspring = list(map(toolbox.clone, toolbox.select(pop, len(pop))))

        # 2. Crossover — repair both children immediately after
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CX_PROB:
                toolbox.mate(child1, child2)
                repair(child1)
                repair(child2)
                del child1.fitness.values
                del child2.fitness.values

        # 3. Mutation — repair the mutant immediately after
        for mutant in offspring:
            if random.random() < MUT_PROB:
                toolbox.mutate(mutant)
                repair(mutant)
                del mutant.fitness.values

        # 4. Re-evaluate only individuals that changed
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)

        # 5. Replace old population
        pop[:] = offspring

        # Track best this generation
        best_now = min(pop, key=lambda x: x.fitness.values[0])
        history.append(best_now.fitness.values[0])

        if gen % 100 == 0:
            print(f"  Gen {gen:5d} | Best: {history[-1]:>10,.0f} | "
                  f"{time.time()-start_time:.1f}s elapsed")

    best_ind = min(pop, key=lambda x: x.fitness.values[0])
    print(f"\nDone — {gen} generations | Best penalty: {best_ind.fitness.values[0]:,.0f}")
    return best_ind, best_ind.fitness.values[0], history

Candidate engines (RUL < 30): 17 → [24, 31, 34, 35, 36, 40, 42, 49, 56, 66, 68, 76, 81, 82, 91, 92, 100]


In [99]:
#Schedule printer and final GA call

def print_schedule(best_ind, rul_input=RUL, T=30):
    """
    Prints the maintenance schedule from the best individual.
    Shows engine, team, type, start/end day, due date, and penalty.
    Returns total penalty cost.
    """
    header = (f"{'Engine':>8} | {'Team':>5} | {'Type':>4} | "
              f"{'Start':>5} | {'End':>5} | {'Due':>5} | {'Penalty':>9}")
    print(header)
    print("─" * len(header))

    maintained = {}
    for engine_id, team_id, start_day in best_ind:
        duration = get_maintenance_duration(engine_id, TEAM_TYPES[team_id])
        end_day  = start_day + duration - 1
        due_date = get_engine_due_date(engine_id, rul_input)
        penalty  = get_penalty_cost(engine_id, end_day, due_date)
        maintained[engine_id] = (team_id, start_day, end_day, due_date, penalty)

        print(f"{engine_id:>8} | {team_id:>5} | {TEAM_TYPES[team_id]:>4} | "
              f"{start_day:>5} | {end_day:>5} | {due_date:>5} | {penalty:>9.0f}")

    total_penalty = sum(v[4] for v in maintained.values())

    # Also report engines that were NOT scheduled but have RUL < T
    print()
    print("Unscheduled engines with RUL < T (incurring full penalty):")
    for engine_id in range(1, 101):
        due = get_engine_due_date(engine_id, rul_input)
        if due < T and engine_id not in maintained:
            penalty = get_penalty_cost(engine_id, T, due)
            total_penalty += penalty
            print(f"  Engine {engine_id}: due={due}, penalty={penalty:.0f}")

    print("─" * len(header))
    print(f"Total penalty cost: {total_penalty:,.0f}")
    return total_penalty


#Run the GA and print the result
best_ind, best_cost, history = run_ga()
print_schedule(best_ind)

  Gen   100 | Best:      4,675 | 2.2s elapsed
  Gen   200 | Best:      3,141 | 4.0s elapsed
  Gen   300 | Best:      2,553 | 5.4s elapsed
  Gen   400 | Best:      2,553 | 6.9s elapsed
  Gen   500 | Best:      2,428 | 8.3s elapsed
  Gen   600 | Best:      2,428 | 9.7s elapsed
  Gen   700 | Best:      2,428 | 11.2s elapsed
  Gen   800 | Best:      2,428 | 12.6s elapsed
  Gen   900 | Best:      2,428 | 14.0s elapsed
  Gen  1000 | Best:      2,428 | 15.5s elapsed
  Gen  1100 | Best:      2,428 | 16.9s elapsed
  Gen  1200 | Best:      2,428 | 18.4s elapsed
  Gen  1300 | Best:      2,428 | 19.8s elapsed
  Gen  1400 | Best:      2,428 | 21.3s elapsed
  Gen  1500 | Best:      2,428 | 22.7s elapsed
  Gen  1600 | Best:      2,428 | 24.1s elapsed
  Gen  1700 | Best:      2,428 | 25.5s elapsed
  Gen  1800 | Best:      2,428 | 26.9s elapsed
  Gen  1900 | Best:      2,428 | 28.3s elapsed
  Gen  2000 | Best:      2,428 | 29.7s elapsed
  Gen  2100 | Best:      2,428 | 31.1s elapsed
  Gen  2200 | Best:

461